# Page 9 — "Outcome vs Process Supervision": two graders on the same chains

One set of reasoning chains, graded twice. An **outcome** grader reads the final answer. A
**process** grader reads every step. This notebook measures what the second one buys, in three
numbers the page has never been able to state:

1. **Detection** — among chains that reached the *right* answer, how many contain a step that is
   demonstrably false. Outcome grading passes every one of them, and page 12's STaR filter would
   fine-tune on every one of them.
2. **Localisation** — among chains that reached the *wrong* answer, how much of the chain was
   sound up to the break. Outcome grading condemns all of it.
3. **Selection** — a process-aware rung on page 8's ladder, if page 8's inputs are on disk.

**Two process graders, and one of them has to earn its labels.** The arithmetic checker is
deterministic: where it judges it is right by construction, and a reader can check any label by
hand. `Qwen2.5-Math-PRM-7B` is a model, so its labels are opinions until measured. Section 6
measures them against the checker on the steps both can see and reports a reliability rather
than claiming one, which is the move page 8 makes with its verifiers.

Writes `out/page_09/`: `traces.jsonl` (chains with per-step labels), `derived.json`,
`manifest.json`.

All rationale lives in [`docs/page_9_refactor.md`](../docs/page_9_refactor.md).


## How to run this

| Mode | Chains | Process grader | Needs |
|---|---|---|---|
| `REPLAY` | page 7's pile, filtered | checker only | no GPU, no weights |
| `QUICK` | 24 generated here | checker only | a laptop CPU, ~30 min |
| `FULL` | 100, page 7's pile if present | checker + `Qwen2.5-Math-PRM-7B` | a 16 GB GPU |

**Which chains this page may grade, and it is a filter rather than an exclusion.** A chain that
was cut off and forced to commit is broken by *us*, so counting that break as the model reasoning
badly would report our own experiment as a finding. Page 6's sweep is capped on purpose and
page 7's *split* axis is budget-starved on purpose, so neither qualifies. Page 7's **pile** does:
those chains run to their natural end, and the ones where the cap never bound are exactly what
this page needs. Section 3 applies the filter and says how many survived it.

The generator is `Qwen2.5-3B-Instruct`, Act 2's shared instrument, so page 9 describes the same
population as the pages before it. The PRM is deliberately *not* from that family: a grader
scoring another model's work has no self-scoring bias to argue about.


In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU), "QUICK" (small, generates), "FULL"
SEED = 123                # selects items; generation here is greedy

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"      # pages 6 and 7's primary, so the pages compare
PRM_MODEL = "Qwen/Qwen2.5-Math-PRM-7B"    # a different family on purpose: no self-scoring

PRESETS = {
    "REPLAY": dict(n_items=100, use_prm=False, may_generate=False),
    "QUICK":  dict(n_items=24,  use_prm=False, may_generate=True),
    "FULL":   dict(n_items=100, use_prm=True,  may_generate=True),
}
CFG = PRESETS[RUN_MODE]
USE_PRM = CFG["use_prm"]

# Page 6's top rung. A safety cap, not a budget: a chain that hits it is one WE cut,
# and section 3 drops those rather than grading them.
CHAIN_CAP = 2048

# Where a step ends. Passed at every call site because it is a claim, not a default
# (harness.step_spans has none). Same value as page 5, so a chain segmented here and
# a chain segmented there break the same way.
MIN_STEP_CHARS = 12

# Below this the PRM is calling a step unsound. Section 6 reports what the threshold
# is worth rather than assuming it.
PRM_THRESHOLD = 0.5

print(f"mode={RUN_MODE}  generator={PRIMARY}")
print(f"process graders: arithmetic checker" + (f" + {PRM_MODEL}" if USE_PRM else " only"))


In [ ]:
import json
import random
import re
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/page_09")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

P7_TRACES = Path("out/page_07/traces.jsonl")
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean, median = harness.mean, harness.median

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])


## 1. The outcome grader

Inherited unchanged from ch03, and it is the grader this whole page is arguing about.
`extract_final_candidate` pulls the last `\boxed{...}`; `grade_answer` checks equivalence.

Note what it can see: the last line. It has no opinion about the hundred tokens before it, which
is the entire subject of this page. The fifth check below is the page in miniature.


In [ ]:
CHECKS = [
    (r"The answer is \boxed{9}.",                        "9",    True),
    (r"27",                                              "9",    False),
    (r"\boxed{\frac{1}{2}}",                             "0.5",  True),
    (r"\boxed{1,024}",                                   "1024", True),
    # right answer via nonsense: the outcome grader says yes, which is the page's subject
    (r"Cancel the 6s: 16/64 -> 1/4. So \boxed{\frac{1}{4}}", "0.25", True),
    (r"I am not sure.",                                  "9",    False),
]

rows = [(raw, gt, exp, bool(grade_answer(extract_final_candidate(raw), gt)))
        for raw, gt, exp in CHECKS]
bad = [r for r in rows if r[3] != r[2]]
if bad:
    for raw, gt, exp, got in rows:
        print(f"{'ok ' if got == exp else 'FAIL'} {extract_final_candidate(raw)!r:<26} "
              f"vs {gt!r:<8} -> {got}")
    raise AssertionError(f"{len(bad)} outcome-grader check(s) behaved unexpectedly")

print(f"outcome grader preflight: {len(rows)}/{len(rows)} as expected")
print("the fifth case is not a bug in the grader. A right answer reached by nonsense is")
print("correct as far as an outcome check can see, and measuring how often that happens")
print("is what this page is for.")


## 2. The item set

MATH-500, levels 4 and 5, page 6's band, so this page and the ones before it describe the same
population.

Two reasons the band matters here specifically. **Wrong answers**: section 8 has nothing to
localise in without them. **Chain length**: a one-line solution cannot show a break partway
through, and easy problems get one-line solutions.


In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

band = sorted((r for r in math500 if int(r["level"]) in (4, 5)),
              key=lambda r: r["unique_id"])          # sort before shuffling
random.Random(SEED).shuffle(band)
items = band[: CFG["n_items"]]
ITEMS_BY_ID = {r["unique_id"]: r for r in items}
ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)

lv = "+".join(f"L{n}x{c}" for n, c in sorted(Counter(int(r["level"]) for r in items).items()))
print(f"{len(items)} items of {len(band)} available at levels 4-5   {lv}   "
      f"sha256 {ITEM_SET_HASH[:12]}")


## 3. Which chains this page is allowed to grade

The filter, and the one design decision here most worth understanding.

Grading a chain for bad reasoning only means something if the reasoning is the model's own. Three
sources, and only some of each qualifies:

| source | qualifies? |
|---|---|
| page 7's **pile** | yes, where `truncated` is false. These run to their natural end. |
| page 7's **split** axis | no. Each chain gets `B/N` tokens by construction. |
| page 6's sweep | no. Capped and force-committed on purpose; that is its experiment. |

So this is a filter on `truncated`, not a blanket refusal to reuse. If nothing survives it and
the mode may generate, the notebook runs its own uncapped pass instead.


In [ ]:
def usable(rec):
    """A chain whose ending was the model's choice, not ours."""
    return (not rec.get("truncated")) and not rec.get("forced") and rec.get("tokens")


source, records = None, []

if P7_TRACES.exists():
    latest = {}
    for line in P7_TRACES.open(encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            latest[(r["run_id"], r.get("config_id"), r["problem_id"],
                    r.get("sample_index", 0))] = r
    pool = [r for r in latest.values() if r["problem_id"] in ITEMS_BY_ID]
    kept = [r for r in pool if usable(r)]

    # One chain per problem, the first by sample index, so no problem is counted twice.
    by_pid = {}
    for r in sorted(kept, key=lambda r: (r["problem_id"], r.get("sample_index", 0))):
        by_pid.setdefault(r["problem_id"], r)

    print(f"page 7: {len(pool):,} records on our items, {len(kept):,} ran to their natural")
    print(f"        end ({len(kept)/max(len(pool),1):.0%}), covering {len(by_pid)} problems")
    if len(by_pid) >= max(8, CFG["n_items"] // 4):
        source = "page_07_pile"
        records = [dict(r, question=ITEMS_BY_ID[r["problem_id"]]["problem"],
                        ground_truth=ITEMS_BY_ID[r["problem_id"]]["answer"],
                        outcome_correct=bool(r["correct"])) for r in by_pid.values()]
    else:
        print("        too few to stand on; generating instead")
else:
    print(f"{P7_TRACES} not found")

if not records and not CFG["may_generate"]:
    raise RuntimeError(
        f"REPLAY has no chains to grade: {P7_TRACES} is absent or nothing in it survived the "
        f"filter. Run page_07_frequency_voting.ipynb, or use QUICK to generate a small pass here.")

print(f"\nsource: {source or 'this notebook (see the next cell)'}")


In [ ]:
COT_SUFFIX = "\n\nSolve this step by step, one step per line."

if not records:
    device = get_device()
    dtype = harness.pick_dtype(device)
    model, tok, n_params = harness.load_model(PRIMARY, device=device, dtype=dtype)
    eos_ids = harness.end_of_turn_ids(tok)

    store = harness.TraceStore(TRACES_PATH, run_id=f"{RUN_MODE.lower()}-{ITEM_SET_HASH[:8]}",
                               key_fields=("problem_id",))
    store.report()

    for row in harness.progress(items, desc="generating", unit="item"):
        hit = store.get(row["unique_id"])
        if hit:
            records.append(hit)
            continue
        g = harness.generate(render_prompt(row["problem"]) + COT_SUFFIX, model, tok,
                             enable_thinking=False, max_new_tokens=CHAIN_CAP, eos_ids=eos_ids)
        extracted = extract_final_candidate(g["text"])
        records.append(store.append({
            "problem_id": row["unique_id"], "level": int(row["level"]),
            "subject": row["subject"], "question": row["problem"],
            "ground_truth": row["answer"], "trace": g["text"], "tokens": g["tokens"],
            "tokens_generated": g["tokens_generated"], "truncated": g["truncated"],
            "extracted_answer": extracted,
            "outcome_correct": bool(grade_answer(extracted, row["answer"])),
        }))

    cut = [r for r in records if r["truncated"]]
    if cut:
        print(f"\ndropping {len(cut)} chain(s) the cap cut: a chain WE ended is not a chain")
        print("the model got wrong, and grading it would report our cap as its error.")
        records = [r for r in records if not r["truncated"]]

    source = "generated_here"
    del model, tok                    # the caller's references are what keep it alive
    harness.free_memory()

n_right = sum(r["outcome_correct"] for r in records)
print(f"\n{len(records)} chains: {n_right} right, {len(records) - n_right} wrong "
      f"({n_right/max(len(records),1):.0%} by the outcome grader)")


## 4. The arithmetic checker

Segmentation comes from `harness.step_spans`, the same rule pages 5 and 7 use, so a chain broken
into steps here breaks the same way there. The checker itself stays in this notebook, because it
is a claim rather than plumbing and a reader has to be able to read it.

### The precision rule, which is the whole design

The checker may judge **only claims that are closed-form numeric on both sides, with no symbols.**
`3/4 = 9/12` is checkable. `Let x = 5` is a definition, not a false claim. `f(2) = 7` depends on a
definition made earlier in the chain. A checker that flagged those as false would be worse than no
checker at all, because the one thing this one has going for it is that its labels are facts a
reader can verify by hand.

Every step therefore gets one of three labels, and the third is not a hole to hide:

| label | meaning |
|---|---|
| `true` | every numeric claim in the step checks out |
| `false` | at least one numeric claim is arithmetically wrong |
| `unknown` | nothing in the step is a closed-form numeric claim |

High precision, low recall, stated openly. The preflight below includes the page's own canonical
example, cancelling the sixes in 16/64, precisely because the checker gets it **wrong in the
generous direction**: the arithmetic is true and the method is nonsense. That is the limit a
deterministic checker cannot cross, and it is why section 6 brings in a model.


In [ ]:
from sympy import simplify, sympify


def _to_sympy_text(s):
    """LaTeX-ish maths to something sympify can read. Deliberately narrow."""
    s = re.sub(r"\\left|\\right", "", s)
    s = re.sub(r"\\[dt]?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", s)
    s = (s.replace("\\times", "*").replace("\\cdot", "*").replace("\\div", "/")
          .replace("^", "**").replace("$", "").replace("\\", ""))
    return re.sub(r"\s+", "", s)


def numeric_value(text):
    """A sympy number if the text is closed-form numeric, else None.

    Any letter disqualifies it, which rejects 'x', 'f(2)', 'pi' and 'sqrt' alike. That
    costs recall and buys the only thing that matters here: no false positives."""
    t = _to_sympy_text(text)
    if not t or not re.search(r"\d", t) or re.search(r"[A-Za-z]", t):
        return None
    try:
        e = sympify(t, rational=True)
    except Exception:
        return None
    return e if getattr(e, "is_number", False) and not e.free_symbols else None


def _strip_prose(side):
    """Drop whole words of prose from the ends of one side of an equation.

    'So 8 * 7' -> '8 * 7' and '56 apples' -> '56': a claim does not stop being checkable
    for having a sentence wrapped round it. Three guards stop this inventing claims that
    were never made, and each one is a false positive that testing actually produced:

    * only tokens of TWO OR MORE letters go. Stripping the 'n' from 'n + 1' would leave
      '+ 1', and 'Let n = 4, so n + 1 = 5' would read as the false claim 1 = 5.
    * 'f(2)' is one token and is not purely alphabetic, so it survives, meets the
      no-letters rule and is skipped, rather than being mined for a '(2)'.
    * if stripping leaves the side starting on an operator we amputated an operand, so
      refuse to judge it at all.
    """
    toks = side.split()
    stripped = False
    while toks and (toks[0].endswith(":")                      # a step label, 'Step 3:'
                    or (toks[0].isalpha() and len(toks[0]) > 1)):
        toks.pop(0)
        stripped = True
    while toks and toks[-1].isalpha() and len(toks[-1]) > 1:
        toks.pop()
    out = " ".join(toks).rstrip(".")
    return "" if stripped and re.match(r"^[+*/^]|^-\s", out) else out


def check_step(text):
    """-> (label, judged_claim). A step is false if ANY numeric claim in it is."""
    # Thousands separators first: splitting on commas below would cut '1,024' in half
    # and turn a valid claim into an unparseable one.
    text = re.sub(r"(?<=\d),(?=\d\d\d\b)", "", text)

    judged = None
    for chunk in re.split(r"[,;]|\bso\b|\bSo\b|\bthus\b|\band\b|\bTherefore\b", text):
        if "=" not in chunk:
            continue
        vals = [numeric_value(_strip_prose(p)) for p in chunk.split("=")]
        if len(vals) < 2 or any(v is None for v in vals):
            continue
        judged = chunk.strip()
        if any(simplify(vals[0] - v) != 0 for v in vals[1:]):
            return "false", judged
    return ("true", judged) if judged else ("unknown", None)


In [ ]:
PREFLIGHT = [
    # --- claims it should judge ------------------------------------------------
    ("3/4 = 9/12",                       "true"),
    ("17/12 - 5/12 = 1",                 "true"),
    ("2 + 2 = 5",                        "false"),
    ("So 8 * 7 = 54.",                   "false"),   # prose in front of the claim
    ("Step 3: 100 - 37 = 62",            "false"),   # a step label in front of it
    ("We get 8 * 7 = 56 apples",         "true"),    # units after it
    ("1,024 = 1024",                     "true"),    # a separator, not a chunk break
    (r"\frac{3}{4} = \frac{9}{12}",      "true"),
    (r"$5 \times 4 = 20$",               "true"),
    ("The total is -5 = -5",             "true"),
    # --- claims it must refuse: each is a false positive waiting to happen ------
    ("Let x = 5",                        "unknown"),  # a definition, not a claim
    ("f(2) = 7",                         "unknown"),  # depends on an earlier definition
    ("Let n = 4, so n + 1 = 5",          "unknown"),  # dropping 'n' would leave '+1 = 5'
    ("Substituting, 2x = 10",            "unknown"),  # an unknown in it
    ("x = 3 and y = 4",                  "unknown"),
    (r"\sqrt{16} = 4",                   "unknown"),  # true, but not closed-form here
    ("(3, 5) = (3, 5)",                  "unknown"),  # a tuple, not arithmetic
    ("The common denominator is 12",     "unknown"),  # no claim at all
    ("The area is 12 square units",      "unknown"),
    # --- the blind spot, as a test rather than as a caveat ---------------------
    ("Cancel the 6s, so 16/64 = 1/4",    "true"),     # arithmetic true, method nonsense
]

fails = 0
for text, expect in PREFLIGHT:
    got, claim = check_step(text)
    if got != expect:
        fails += 1
        print(f"FAIL {text!r:<38} -> {got:<8} expected {expect}")
assert not fails, f"{fails} of {len(PREFLIGHT)} checker preflight cases behaved unexpectedly"

print(f"checker preflight: {len(PREFLIGHT)}/{len(PREFLIGHT)} as expected")
print(f"  {sum(1 for _, e in PREFLIGHT if e != 'unknown')} it judges, "
      f"{sum(1 for _, e in PREFLIGHT if e == 'unknown')} it refuses")
print("\nThe last case is the one to notice: 'cancel the 6s, so 16/64 = 1/4' comes back TRUE.")
print("The arithmetic is right and the method is nonsense, and no deterministic checker can")
print("tell those apart. That gap is what section 6 is for.")


In [ ]:
for r in harness.progress(records, desc="checking steps", unit="chain"):
    steps = harness.segment_steps(r["trace"], r["tokens"], min_step_chars=MIN_STEP_CHARS)
    for s in steps:
        s["check"], s["claim"] = check_step(s["text"])
    r["steps"] = steps
    r["n_steps"] = len(steps)
    r["first_false"] = next((i for i, s in enumerate(steps) if s["check"] == "false"), None)
    r["n_false"] = sum(s["check"] == "false" for s in steps)
    r["n_checkable"] = sum(s["check"] in ("true", "false") for s in steps)

all_steps = [s for r in records for s in r["steps"]]
lab = Counter(s["check"] for s in all_steps)

harness.table(
    "step labels",
    ["label", "steps", "share"],
    [[k, lab[k], f"{lab[k]/max(len(all_steps),1):.1%}"] for k in ("true", "false", "unknown")],
    note=f"{len(all_steps):,} steps over {len(records)} chains; "
         f"median {median([r['n_steps'] for r in records]):.0f} steps per chain")

print(f"\nchains with at least one step the checker could judge: "
      f"{sum(r['n_checkable'] > 0 for r in records)}/{len(records)}")
if lab["unknown"] / max(len(all_steps), 1) > 0.8:
    print("Most steps are beyond a deterministic checker, which is itself the argument for")
    print("section 6 and worth saying on the page rather than hiding.")


## 5. The process reward model

The checker cannot see method errors. A model can, and `Qwen2.5-Math-PRM-7B` emits a score per
reasoning step, which is process supervision in the literal sense.

But a PRM's labels are opinions, and putting them on the page unexamined would repeat exactly the
defect this refactor exists to remove: the current page reveals hand-written booleans and calls
them a PRM's verdict. So the PRM does not get to be the authority here. It gets **measured**, in
section 6, against the checker on the steps both can see, because on its own subset the checker
is right by construction.

If the PRM is off, will not load, or fails the separation preflight, everything below still runs
from the checker alone and `derived.json` records that it did not participate.


In [ ]:
PRM_READY, prm, prm_tok = False, None, None
STEP_SEP = "<extra_0>"          # the token this PRM scores at, one per step

if USE_PRM:
    try:
        from transformers import AutoModel
        device = get_device()
        dtype = harness.pick_dtype(device)
        print(f"[1/2] fetch      {PRM_MODEL}")
        prm_path = harness.fetch(PRM_MODEL)
        print("[2/2] load       PRM weights")
        prm_tok = AutoTokenizer.from_pretrained(prm_path, trust_remote_code=True)
        prm = AutoModel.from_pretrained(prm_path, dtype=dtype,
                                        trust_remote_code=True).to(device).eval()
        PRM_READY = True
    except Exception as e:
        print(f"\nPRM unavailable ({type(e).__name__}: {e})")
        print("Continuing with the checker alone; sections 7 and 8 still produce their numbers.")
else:
    print(f"USE_PRM is off in {RUN_MODE}. The checker carries every measurement below.")


@torch.inference_mode()
def prm_scores(question, step_texts):
    """One score in [0,1] per step. Higher means the PRM thinks the step is sound."""
    body = STEP_SEP.join(t.strip() for t in step_texts) + STEP_SEP
    chat = prm_tok.apply_chat_template(
        [{"role": "system", "content": "Please reason step by step."},
         {"role": "user", "content": question},
         {"role": "assistant", "content": body}],
        tokenize=False, add_generation_prompt=False)
    enc = prm_tok(chat, return_tensors="pt").to(prm.device)
    out = prm(**enc)
    logits = out[0] if isinstance(out, (tuple, list)) else getattr(out, "logits", out)
    sep_id = prm_tok.encode(STEP_SEP, add_special_tokens=False)[0]
    probs = torch.softmax(logits.float(), dim=-1)[enc["input_ids"] == sep_id]
    return probs[:, -1].tolist()


if PRM_READY:
    # A clean chain against the same chain with one step corrupted. If the PRM cannot
    # separate those, its scores are noise and nothing below should use them.
    q = "What is 3/4 + 2/3?"
    clean = ["The common denominator is 12.", "3/4 = 9/12 and 2/3 = 8/12.",
             "9/12 + 8/12 = 17/12.", "So the answer is \\boxed{17/12}."]
    dirty = list(clean)
    dirty[2] = "9/12 + 8/12 = 15/12."                    # now arithmetically false

    a, b = prm_scores(q, clean), prm_scores(q, dirty)
    assert len(a) == len(clean), (
        f"PRM returned {len(a)} scores for {len(clean)} steps; the separator convention "
        f"for this checkpoint is not what this cell assumes")

    harness.table("PRM separation preflight", ["step", "clean", "corrupted"],
                  [[i + 1, f"{x:.3f}", f"{y:.3f}" + ("   <- corrupted" if i == 2 else "")]
                   for i, (x, y) in enumerate(zip(a, b))])

    if b[2] >= a[2]:
        PRM_READY = False
        print("\nIt does not mark a known-false step lower than a true one, so its scores are")
        print("not usable as labels. Falling back to the checker alone.")


In [ ]:
if PRM_READY:
    for r in harness.progress(records, desc="PRM scoring", unit="chain"):
        try:
            sc = prm_scores(r["question"], [s["text"] for s in r["steps"]])
        except Exception:
            sc = []
        r["prm_ok"] = len(sc) == len(r["steps"])
        if not r["prm_ok"]:
            continue
        for s, v in zip(r["steps"], sc):
            s["prm"] = float(v)
            s["prm_label"] = "false" if v < PRM_THRESHOLD else "true"
        r["prm_first_false"] = next(
            (i for i, s in enumerate(r["steps"]) if s["prm_label"] == "false"), None)
        r["prm_n_false"] = sum(s["prm_label"] == "false" for s in r["steps"])
    print(f"\nPRM scored {sum(r.get('prm_ok', False) for r in records)}/{len(records)} chains")
else:
    for r in records:
        r["prm_ok"] = False


## 6. Is the PRM worth believing?

On steps the checker judged, how often does the PRM agree? Where they disagree the checker is
right, by construction, so this is a measurement of the PRM and not a negotiation between two
opinions.

Three numbers, and they answer different questions. **Agreement** is the headline. **Recall on
false steps** is whether it catches breaks the checker proved were there. **Specificity** is
whether it cries wolf on steps that were fine, which matters more than it looks: a grader that
flags a third of correct steps would make section 8's localisation meaningless.

Whatever comes out here has to be attached to any PRM-only claim the page makes about the steps
the checker could not reach.


In [ ]:
RELIABILITY = None
pairs = [(s["check"], s["prm_label"]) for r in records if r.get("prm_ok")
         for s in r["steps"] if s["check"] in ("true", "false") and "prm_label" in s]

if pairs:
    f_steps = [p for p in pairs if p[0] == "false"]
    t_steps = [p for p in pairs if p[0] == "true"]
    RELIABILITY = {
        "n_steps_compared": len(pairs),
        "agreement": mean([a == b for a, b in pairs]),
        "recall_on_false": mean([b == "false" for _, b in f_steps]) if f_steps else None,
        "specificity_on_true": mean([b == "true" for _, b in t_steps]) if t_steps else None,
        "n_false_steps": len(f_steps),
        "threshold": PRM_THRESHOLD,
    }
    harness.table(
        "PRM against the arithmetic checker",
        ["measure", "value", "over"],
        [["agreement", f"{RELIABILITY['agreement']:.1%}", f"{len(pairs)} steps"],
         ["recall on false steps",
          f"{RELIABILITY['recall_on_false']:.1%}" if f_steps else "n/a", f"{len(f_steps)} steps"],
         ["specificity on true steps",
          f"{RELIABILITY['specificity_on_true']:.1%}" if t_steps else "n/a",
          f"{len(t_steps)} steps"]],
        note="where they disagree the checker is right, by construction")
elif PRM_READY:
    print("no overlapping steps to compare on, so the PRM's reliability is unmeasured and")
    print("the page may not report PRM-only judgements")
else:
    print("no PRM in this run; every measurement below is the checker's")


## 7. Detection: how often does outcome grading reward nonsense

Spec §3a. Among chains that reached the **right** answer, the share containing a step the checker
proved false. Outcome grading passes every one.

**This may come out small, and the spec says so in advance.** A false step usually propagates
into a wrong answer, and the cases where it does not are often method errors with valid
arithmetic, which the checker cannot see. If it is small, the page reports it as small and
changes its verb rather than inflating it.

Small still matters, because of where it lands. Page 12 is STaR: generate rationales, keep the
ones that reach the right answer, fine-tune on those. This share is the contamination rate of
that filter.


In [ ]:
right = [r for r in records if r["outcome_correct"]]
wrong = [r for r in records if not r["outcome_correct"]]
right_bad = [r for r in right if r["n_false"] > 0]

DETECTION = {
    "grader": "arithmetic checker", "source": source,
    "n_right": len(right), "n_right_with_false_step": len(right_bad),
    "base_rate": len(right_bad) / max(len(right), 1),
    "n_wrong": len(wrong),
}

rows = [["outcome grader", f"{len(right)} of {len(records)} chains pass", ""],
        ["of those, a step the checker proved false", len(right_bad),
         f"{DETECTION['base_rate']:.1%}"]]

if PRM_READY and RELIABILITY:
    pr = [r for r in right if r.get("prm_ok")]
    pr_bad = [r for r in pr if r.get("prm_n_false", 0) > 0]
    DETECTION["prm_base_rate"] = len(pr_bad) / max(len(pr), 1)
    DETECTION["prm_agreement"] = RELIABILITY["agreement"]
    rows.append(["of those, a step the PRM calls unsound", len(pr_bad),
                 f"{DETECTION['prm_base_rate']:.1%}"])

harness.table("detection", ["", "count", "share of right answers"], rows,
              note=("the PRM row also sees non-arithmetic steps; attach its measured "
                    f"{RELIABILITY['agreement']:.0%} agreement to it"
                    if PRM_READY and RELIABILITY else
                    "the checker sees arithmetic only, so this is a floor rather than a rate"))

if DETECTION["base_rate"] == 0:
    print("\nZERO -- a fact about the checker before it is a fact about the model. See the")
    print("spec's open question 4 before reporting this as a finding about reasoning.")
elif DETECTION["base_rate"] < 0.05:
    print("\nSmall. Report it as small; the STaR framing is what gives it stakes.")


## 8. Localisation: how much of a wrong chain was sound

Spec §3b, and the measurement Solution B on the current page has always taught without evidence.

Take chains that reached the **wrong** answer where the checker finds a break at step *k* of *n*.
Outcome grading can only say *wrong*, which condemns all *n*. Process grading points at *k* and
leaves the first *k-1* standing. So the quantity is `k/n`, the share of the chain that survives.

The second reading tests something the page asserts rather than shows: whether a break really is
one step, or whether real chains carry several.


In [ ]:
localisable = [r for r in wrong if r["first_false"] is not None and r["n_steps"] > 1]
LOCALISATION = {"n_wrong": len(wrong), "n_localisable": len(localisable)}

if localisable:
    shares = sorted(r["first_false"] / r["n_steps"] for r in localisable)
    med = shares[len(shares) // 2]
    dist = Counter(r["n_false"] for r in localisable)
    LOCALISATION.update({
        "median_share_sound": med,
        "mean_share_sound": mean(shares),
        "share_with_one_false_step": dist[1] / len(localisable),
        "n_false_distribution": dict(sorted(dist.items())),
    })

    harness.table(
        "what each grader condemns, on a wrong chain with a findable break",
        ["grader", "verdict", "share of the chain condemned"],
        [["outcome", "wrong", "100%"],
         ["process", f"broken at step {med:.0%} of the way in", f"{1 - med:.0%}"]],
        note=f"{len(localisable)} of {len(wrong)} wrong chains have a break the checker can find")

    print(f"\n-> the first {med:.0%} of the chain was sound, and outcome grading threw it away")
    print(f"\nbreaks per chain: {LOCALISATION['n_false_distribution']}")
    print(f"exactly one broken step: {LOCALISATION['share_with_one_false_step']:.0%}")
    if LOCALISATION["share_with_one_false_step"] < 0.5:
        print("  ^ under half, so the page must NOT claim breaks localise to a single line.")
        print("    Follow the data: say how many, and keep Solution B as one case, not the rule.")
else:
    print(f"No wrong chain has a findable break, out of {len(wrong)} wrong chains.")
    print("Section 8 has nothing to plot. Either the checker's recall is too low on this")
    print("model's output, or these answers fail for non-arithmetic reasons.")


In [ ]:
import matplotlib.pyplot as plt

if localisable:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2),
                                   gridspec_kw={"width_ratios": [3, 2]})
    ax1.hist([s * 100 for s in shares], bins=10, range=(0, 100), color="#3b4a6b")
    ax1.axvline(med * 100, color="#c2703d", lw=2)
    ax1.set_xlabel("share of the chain before the break (%)")
    ax1.set_ylabel("wrong chains")
    ax1.set_title("where the break lands", fontsize=10, loc="left")

    ax2.barh([1, 0], [100, (1 - med) * 100], height=0.5, color=["#8c8c8c", "#3b4a6b"])
    ax2.set_yticks([1, 0]); ax2.set_yticklabels(["outcome", "process"])
    ax2.set_xlim(0, 100); ax2.set_xlabel("% of the chain condemned")
    ax2.set_title("what each grader throws away", fontsize=10, loc="left")

    for ax in (ax1, ax2):
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"{PRIMARY} · MATH-500 L4-5 · n={len(localisable)} wrong chains with a "
                 f"findable break", fontsize=9, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_localisation.png", dpi=160, bbox_inches="tight")
    plt.show()


## 9. Selection: a process-aware rung on page 8's ladder

Spec §3c. Page 8 ranks selectors that all read the final answer; this adds one that reads the
working. Among a problem's candidates, drop any containing a checker-false step, then take the
majority of what survives.

**Optional and self-skipping.** It needs page 7's candidate pile. If that is absent this reports
so and the page ships sections 7 and 8 alone, which are what it leads with anyway.

Expect a small gain, and report whatever comes out. A null result is a real finding about how
much of the failure is method rather than arithmetic, and it is one page 8's ladder is already
shaped to show.


In [ ]:
SELECTION = None

if not P7_TRACES.exists():
    print(f"{P7_TRACES} not found -- skipping the selector rung. Sections 7 and 8 stand alone.")
else:
    from collections import defaultdict
    cands = defaultdict(list)
    seen = {}
    for line in P7_TRACES.open(encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            seen[(r["run_id"], r.get("config_id"), r["problem_id"],
                  r.get("sample_index", 0))] = r
    for r in seen.values():
        if usable(r):
            cands[r["problem_id"]].append(r)
    cands = {k: v for k, v in cands.items() if len(v) >= 4}

    if not cands:
        print("page 7's log has no problem with four or more natural-length candidates;")
        print("skipping. This needs its pile axis, not its split axis.")
    else:
        def majority(rs):
            t = Counter(r["extracted_answer"] for r in rs if r["extracted_answer"] is not None)
            return t.most_common(1)[0][0] if t else None

        plain, filtered, emptied = [], [], 0
        for pid, rs in harness.progress(sorted(cands.items()), desc="selecting", unit="item"):
            truth = rs[0]["ground_truth"]
            keep = [r for r in rs
                    if not any(check_step(s["text"])[0] == "false" for s in
                               harness.segment_steps(r["trace"], r["tokens"],
                                                     min_step_chars=MIN_STEP_CHARS))]
            if not keep:
                emptied += 1
                keep = rs                    # dropping everything is not a selection
            plain.append(bool(grade_answer(majority(rs), truth)))
            filtered.append(bool(grade_answer(majority(keep), truth)))

        SELECTION = {
            "n_problems": len(cands),
            "majority": mean(plain),
            "majority_after_step_filter": mean(filtered),
            "delta_pts": (mean(filtered) - mean(plain)) * 100,
            "n_problems_all_candidates_filtered": emptied,
        }
        harness.table(
            "selection, on page 7's pile",
            ["rule", "reads", "accuracy"],
            [["majority vote", "the answer", f"{SELECTION['majority']:.1%}"],
             ["majority after dropping chains with a false step", "the working",
              f"{SELECTION['majority_after_step_filter']:.1%}"]],
            note=f"{SELECTION['delta_pts']:+.1f} points over {len(cands)} problems; "
                 f"{emptied} had every candidate filtered and fell back to the full set")

        if abs(SELECTION["delta_pts"]) < 1:
            print("\nEssentially null. Say so: at this checker's recall, reading the working")
            print("buys almost nothing at selection time, and detection and localisation are")
            print("where step grading earns its keep.")


## 10. The three specimens

The page's interactive keeps its structure: the reader marks each step, then the labels are
revealed. What changes is that the chains are real and the labels are the checker's, so the
reveal is a fact rather than an attribution to a system that never ran.

| role | what it shows |
|---|---|
| `lucky` | right answer, at least one false step. Outcome grading rewards it. |
| `broken` | wrong answer, one findable break. Outcome says *wrong*; process says *step k*. |
| `clean` | right answer, nothing false. The graders agree. |

Curated by filter, shortest first. A role that cannot be filled is reported rather than faked,
and a missing role blocks translation for that panel: an authored trace must not be substituted.


In [ ]:
def pick(pool):
    ranked = sorted(pool, key=lambda r: (r["n_steps"], r["problem_id"]))
    return next((r for r in ranked if 2 <= r["n_steps"] <= 8), ranked[0] if ranked else None)


SPECIMENS, missing = {}, []
for role, pool in {
    "lucky":  [r for r in right if r["n_false"] > 0],
    "broken": [r for r in wrong if r["n_false"] == 1 and r["n_steps"] > 1],
    "clean":  [r for r in right if r["n_false"] == 0 and r["n_checkable"] > 0],
}.items():
    got = pick(pool)
    if got is None:
        missing.append(role)
        continue
    SPECIMENS[role] = {k: got[k] for k in (
        "problem_id", "level", "question", "ground_truth", "extracted_answer",
        "outcome_correct", "trace", "tokens", "steps", "first_false", "n_false")}
    SPECIMENS[role]["role"] = role
    SPECIMENS[role]["prm_scored"] = got.get("prm_ok", False)
    marks = " ".join({"true": "+", "false": "X", "unknown": "?"}[s["check"]] for s in got["steps"])
    print(f"{role:<7} {got['problem_id']:<30} {got['n_steps']:>2} steps   {marks}")

print("\n  + checked true    X checked false    ? no numeric claim to check")

if missing:
    print(f"\nMISSING ROLES: {missing}. derived.json records this. A role with no specimen")
    print("cannot be translated to the page; do not substitute an authored trace for it.")

if "broken" in SPECIMENS:
    sp = SPECIMENS["broken"]
    print(f"\n--- {sp['problem_id']}: the localisation case " + "-" * 28)
    print(textwrap.fill("Q: " + sp["question"], 92))
    for i, s in enumerate(sp["steps"]):
        arrow = "->" if s["check"] == "false" else "  "
        print(f"{arrow} {i+1}. [{s['check']:<7}] {s['text'][:84]}")
    print(f"\noutcome grader: WRONG (said {sp['extracted_answer']}, truth {sp['ground_truth']}),")
    print(f"                which condemns all {len(sp['steps'])} steps")
    print(f"process grader: step {sp['first_false'] + 1}; the {sp['first_false']} before it stand")


## 11. Artifacts

`derived.json` is the contract with the page: it reads that and nothing else, and no inference
runs at demonstration time.

The manifest records something the other pages do not need: **which grader produced which
label**, and how well the model-based one did on the steps we could check independently. A
reader disputing a step label has to be able to find out who made it.


In [ ]:
manifest = {
    "run_id": f"{RUN_MODE.lower()}-{ITEM_SET_HASH[:8]}",
    "notebook": "page_09_outcome_vs_process.ipynb",
    "page": 9,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "harness": harness.fingerprint(),
    "chains": {
        "source": source,
        "n": len(records),
        "filter": "truncated is false and not forced -- a chain WE ended is not the "
                  "model's reasoning, and grading it would report our cap as its error",
        "generator": PRIMARY,
        "generator_revision": harness.repo_revision(PRIMARY),
        "chain_cap": CHAIN_CAP,
        "prompt_suffix": COT_SUFFIX,
    },
    "graders": {
        "outcome": "reasoning_from_scratch.ch03.grade_answer",
        "arithmetic_checker": {
            "rule": "closed-form numeric on both sides, no symbols; true/false/unknown",
            "preflight_cases": len(PREFLIGHT),
            "known_limit": "cannot see method errors that use valid arithmetic",
        },
        "prm": ({"id": PRM_MODEL, "revision": harness.repo_revision(PRM_MODEL),
                 "threshold": PRM_THRESHOLD, "reliability_vs_checker": RELIABILITY}
                if PRM_READY else None),
    },
    "items": {"source": "MATH-500 (ch03 copy)", "filter": "level in {4, 5}",
              "n": len(items), "item_set_sha256": ITEM_SET_HASH, "seed": SEED},
    "step_segmentation": {"via": "harness.step_spans", "min_step_chars": MIN_STEP_CHARS},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__},
}

derived = {
    "manifest": manifest,
    "detection": DETECTION,
    "localisation": LOCALISATION,
    "selection": SELECTION,
    "prm_reliability": RELIABILITY,
    "step_label_counts": dict(lab),
    "specimens": SPECIMENS,
    "missing_roles": missing,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
with TRACES_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

for p, tier in ((DERIVED_PATH, "bundled"), (MANIFEST_PATH, "bundled"),
                (TRACES_PATH, "download")):
    print(f"{p.name:<16} {harness.kb(p):>9.1f} KB   <- {tier}")


## Notes for re-derivation

- The checker is **high precision, low recall by design**. Every `false` is verifiable by hand;
  every `unknown` is a step it declined to judge, not one it approved.
- Detection and localisation come from the checker alone, so they stand whether or not the PRM
  ran. Any PRM-only claim carries the reliability from section 6.
- The chains are filtered to those the model ended itself. That filter, not the source, is what
  makes them gradable; page 7's pile qualifies and its split axis does not.
- Greedy decoding still diverges across hardware on a few items. Expect the shape to reproduce,
  not every digit.

What the page may and may not claim is [`docs/page_9_refactor.md`](../docs/page_9_refactor.md),
which the translation step checks itself against
([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §6).
